In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device="cuda" if torch.cuda.is_available() else "cpu"
print(device)

# HYPERPARAMETERS
block_size=8
batch_size=4
max_iters=1000
#eval_interval=2500
learning_rate=3e-4
eval_iters=250

cuda


In [2]:
with open ("wizard-of-oz.txt","r",encoding="utf-8")as f:
    text=f.read()
chars=sorted(set(text))
print(len(chars))
vocab_size=len(chars)

80


In [3]:
string_to_int={ch:i for i,ch in enumerate(chars)}
int_to_string={i:ch for i,ch in enumerate(chars)}
encode=lambda s: [string_to_int[c] for c in s]
decode=lambda l: ''.join([int_to_string[i] for i in l])

data=torch.tensor(encode(text),dtype=torch.long)
print(data[:100])

tensor([ 1,  1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,
         1, 47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26,
        49,  0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,
         0,  0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1,
        36, 25, 38, 28,  1, 39, 30,  1, 39, 50])


In [4]:
n=int(0.8*len(data))
train_data=data[:n]
val_data=data[n:]

def get_batch(split):
    data=train_data if split=="train" else val_data
    ix=torch.randint(len(data)-block_size,(batch_size,))
    #print(ix)
    x=torch.stack([data[i:i+block_size] for i in ix])
    y=torch.stack([data[i+1:i+block_size+1] for i in ix])
    x,y=x.to(device),y.to(device)
    return x,y

x,y=get_batch("train")
print("Inputs:")
print(x)
print("Targets:")
print(y)

Inputs:
tensor([[67, 58,  1, 68, 59,  1, 73, 61],
        [ 9,  1, 28, 68, 71, 68, 73, 61],
        [73, 58, 54, 71, 72,  1, 33,  1],
        [54, 71, 68, 74, 67, 57,  1, 61]], device='cuda:0')
Targets:
tensor([[58,  1, 68, 59,  1, 73, 61, 58],
        [ 1, 28, 68, 71, 68, 73, 61, 78],
        [58, 54, 71, 72,  1, 33,  1, 76],
        [71, 68, 74, 67, 57,  1, 61, 62]], device='cuda:0')


In [5]:
x=train_data[:block_size]
y=train_data[1:block_size+1]

for t in range(block_size):
    context=x[:t+1]
    target=y[t]
    print("when input is",context,"target is ", target) 

when input is tensor([1]) target is  tensor(1)
when input is tensor([1, 1]) target is  tensor(1)
when input is tensor([1, 1, 1]) target is  tensor(28)
when input is tensor([ 1,  1,  1, 28]) target is  tensor(39)
when input is tensor([ 1,  1,  1, 28, 39]) target is  tensor(42)
when input is tensor([ 1,  1,  1, 28, 39, 42]) target is  tensor(39)
when input is tensor([ 1,  1,  1, 28, 39, 42, 39]) target is  tensor(44)
when input is tensor([ 1,  1,  1, 28, 39, 42, 39, 44]) target is  tensor(32)


In [6]:
@torch.no_grad()
def estimate_loss():
    out={}
    model.eval()
    for split in ["train","val"]:
        losses=torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y=get_batch(split)
            logits,loss=model(X,Y)
            losses[k]=loss.item()
        out[split]=losses.mean()
    model.train()
    return out

In [11]:
class GPTLanguageModel(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,vocab_size)

    def forward(self,index,targets=None):
        logits=self.token_embedding_table(index)
        if targets is None:
            loss=None
        else:
            B,T,C=logits.shape
            logits=logits.view(B*T,C)
            targets=targets.view(B*T)
            loss=F.cross_entropy(logits,targets)
        return logits, loss

    def generate(self,index,max_new_tokens):
        for _ in range(max_new_tokens):
            logits,loss=self.forward(index)
            logits=logits[:,-1,:]
            probs=F.softmax(logits,dim=-1)
            index_next=torch.multinomial(probs,num_samples=1)
            index=torch.cat((index,index_next),dim=1)
        return index

model=GPTLanguageModel(vocab_size)
m=model.to(device)

context=torch.zeros((1,1),dtype=torch.long,device=device)
generated_chars=decode(m.generate(context,max_new_tokens=500)[0].tolist())
print(generated_chars)


Efv
Bve,qx9Cmn)ebDck7C)9tiJ_8*vdMJ:fQ&YG GLil2l[q1PEOl 7?*!5KW*nJtww Gc0KIYMjp;MD2VL?TwJn7GZ'8KWiM.Kt9v398RD9q*v0Yy7QESa0O?2pA8Kg((Wh9 JIRn_(a"MfA8Wn;SqxGYWF'Xs,e7Pq3YH[f5t4y?qVja.d;;0KVav.9_*A,lnMeBB
JGQ?zQxi2OE,e-]9K[Sc;"f.gyeM)LzOE"8OG eMGP))H3zgE."& 8n;[mIsE-3F,]:7!*TVaiqfb9k;dv"!ItQYk1-x96J
NKYg7:uS*i0jktX F,l'O)ey!ST:tmW*Ei2(N8iatt7Vf9DR9:usO]KI3g1gGX9 8a3*2t!D*0!4uAwsS296ife8;_2_7Gqiy;ox6962:KpLi85wN8kvB&_6 Y;S*rg_gl(pNy;raOI!3PeAZ]yoB AO1SlYBBavo4_Xb)Hj0W:r;mw.qn;]6m)W[5,l*uT8k75KDLna0!C


In [8]:
optimizer=torch.optim.AdamW(model.parameters(),lr=learning_rate)

for iter in range(max_iters):
    if iter%eval_iters==0:
        losses=estimate_loss()
        print(f"step:{iter},train loss {losses["train"]:.3f}, val loss: {losses["val"]:.3f}")
        
    xb,yb=get_batch("train")
    logits,loss=model.forward(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step:0,train loss 4.773, val loss: 4.783
step:250,train loss 4.717, val loss: 4.739
step:500,train loss 4.662, val loss: 4.654
step:750,train loss 4.587, val loss: 4.597
4.516417980194092


In [9]:
context=torch.zeros((1,1),dtype=torch.long,device=device)
generated_chars=decode(m.generate(context,max_new_tokens=500)[0].tolist())
print(generated_chars)


5!8)Yz1Xq7gfa[;7ZuHVpfd7H?e3QLny9'1CjwAnbI
VsruiBFnjFqW-td007niUllUk7nCd9i.-T2o4GYzFFeH
SkVv
p?3z[kU&H[&W'-oiE4K6AnlON-.0J]obhVamm(DlSylk3l30 Ai!-o!L1P]S*D
c8*28gkZP1zP1gAny1cRlGH1*OtFq8vT6hW-YTWQG0*gfGstOwKuE6G cFUW4'Ta9-on2Ann9f"an9[qu0jGEi.
,&9v9Dv)MVI&zEc:5awnRj*EPRdOzgA3.O'1tJ5lsrMntHVMHyPP[81Vy Ayut"VpHd!9pRBzd7Cr1LR)Aq&b"M_p78P91H63fiX9[qX)TLBzTpH?U&ny"SABCC4'tWWf'1 An._[NS DBU(:m_tH87WQVdXK"ZKEawKQ;Sd)!k3z]rsgFn9li.,pkTaFJt!]3f*']HqXuI[.Pc!M6I4CuMrec !KFvKcnRGG93&HVre?Yz?-)hh5im dPPP]Iw?
